# 03 · 因子检验：IC 分析与分层回测

> 本 notebook 是《量化研究入门学习资料》第 X 章的可运行配套。
> 数据源：`data/csv/`。运行前请先执行 `python scripts/generate_data.py` 生成数据。

## 目标
从 `factors.csv` 重算：月度 IC / Rank IC / ICIR / t 值（对照 `ic.json`）与十分位分层净值（对照 `layers.json`）。

In [ ]:
import pandas as pd, numpy as np
import json

factors = pd.read_csv("data/csv/factors.csv", parse_dates=["date"])
FACTORS = ["EP", "SIZE", "MOM60", "REV5", "VOL20", "TURN", "ROE", "GROW"]

# ---- 月度 IC / Rank IC ----
def measure_ic(factors):
    ic, ric = {f: [] for f in FACTORS}, {f: [] for f in FACTORS}
    for _, g in factors.groupby("date"):
        y = g["next_return"]
        for f in FACTORS:
            x = g["z_" + f]
            m = x.notna() & y.notna()
            if m.sum() >= 10:
                ic[f].append(np.corrcoef(x[m], y[m])[0, 1])
                ric[f].append(pd.Series(x[m]).corr(pd.Series(y[m]), method="spearman"))
    summ = {}
    for f in FACTORS:
        a = np.array(ic[f]); sd = a.std() if len(a) > 1 else 0
        summ[f] = dict(mean=float(a.mean()), std=float(sd),
                       icir=float(a.mean() / sd) if sd else None,
                       t=float(a.mean() / (sd / np.sqrt(len(a)))) if sd else None)
    return ic, ric, summ

ic, ric, summ = measure_ic(factors)
for f in FACTORS:
    print(f"{f:6s} IC={summ[f]['mean']:+.4f}  ICIR={summ[f]['icir']:.3f}  t={summ[f]['t']:6.2f}")

In [ ]:
# ---- 对照 ic.json ----
ref = json.load(open("data/ic.json", encoding="utf-8"))
ok_ic = all(np.allclose(np.array(ic[f]), np.array(ref["ic"][f]), atol=1e-5) for f in FACTORS)
ok_s = all(abs(summ[f]["mean"] - ref["summary"][f]["mean"]) < 5e-5 for f in FACTORS)
print("IC 序列对照:", "PASS" if ok_ic else "FAIL")
print("IC 均值对照:", "PASS" if ok_s else "FAIL")
assert ok_ic and ok_s

In [ ]:
# ---- 十分位分层 ----
def build_layers(factors):
    dates = sorted(factors["date"].unique())
    res = {}
    for f in FACTORS:
        nav = {f"L{k}": [1.0] for k in range(1, 11)}
        ls, monthly = [1.0], {f"L{k}": [] for k in range(1, 11)}
        for _, g in factors.groupby("date"):
            x, y = g["z_" + f], g["next_return"]
            m = x.notna() & y.notna()
            if m.sum() < 20:
                for k in range(1, 11):
                    nav[f"L{k}"].append(nav[f"L{k}"][-1]); monthly[f"L{k}"].append(np.nan)
                ls.append(ls[-1]); continue
            q = pd.qcut(x[m], 10, labels=False, duplicates="drop")
            for k in range(10):
                r = float(y[m][q == k].mean())
                if not np.isfinite(r): r = 0.0    # 空层（分层退化）按收益 0 处理
                monthly[f"L{k+1}"].append(r)
                nav[f"L{k+1}"].append(nav[f"L{k+1}"][-1] * (1 + r))
            ls.append(ls[-1] * (1 + monthly["L10"][-1] - monthly["L1"][-1]))
        res[f] = {**{k: v for k, v in nav.items()}, "LS": ls}
    return res

layers = build_layers(factors)
ref_l = json.load(open("data/layers.json", encoding="utf-8"))
ok_l = all(np.allclose(np.array(layers[f][f"L{k}"][1:]), np.array(ref_l["nav"][f][f"L{k}"][1:]), atol=1e-5)
           for f in FACTORS for k in range(1, 11))
print("分层净值对照:", "PASS" if ok_l else "FAIL")
assert ok_l
print("\nEP 分层多空累计收益:", round(layers["EP"]["LS"][-1], 3))